In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Import all imp library

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import wandb
import random, warnings



warnings.filterwarnings('ignore')

seed = 42
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print("Device Name: ", torch.cuda.get_device_name())
print("GPU: ", device)

# Load Datasets

In [ ]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [ ]:
train_df.head()

In [ ]:
test_df.head()

# EDA

In [ ]:
print(f'Train shape: {train_df.shape}, Test Shape: {test_df.shape}')
print(f'Train Columns {list(train_df.columns)}')
print(f'Test Columns: {list(test_df.columns)}')
print(f'Train Dtype: {train_df.dtypes}, Test Dtype: {test_df.dtypes}')

In [ ]:
print("Null Values")
print(f'Train Null Values: {train_df.isnull().sum()}')
print(f'Test Null Values: {test_df.isnull().sum()}')

In [ ]:
# answer distribution
print(train_df.answer.value_counts().sort_index())

# Preprocessing

In [ ]:
train_df_unique = train_df.drop_duplicates(subset=["prompt"]).copy()

def combine_mcq(row):
    return f"Question: {row['prompt']} Options: A) {row['A']} B) {row['B']} C) {row['C']} D) {row['D']} E) {row['E']}"

train_df_unique['text'] = train_df_unique.apply(combine_mcq, axis=1)
test_df['text'] = test_df.apply(combine_mcq, axis=1)

In [ ]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df_unique['text'], train_df_unique['answer'], 
    test_size=0.2, random_state=42, stratify=train_df_unique['answer']
)

print(f"Training samples: {len(train_texts)}")
print(f"Validation samples: {len(val_texts)}")

le = LabelEncoder()
train_labels_enc = le.fit_transform(train_labels)
val_labels_enc = le.transform(val_labels)

In [ ]:
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB_API_KEY")

wandb.login(key=wandb_key)

# Built model from scratch

In [ ]:
# Model 1 Built from scratch
wandb.init(project="23f3003059-t22026", name="MLP_Scratch", config={"metric": "MAP@3"})

tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = tfidf.fit_transform(train_texts)
X_val_tfidf = tfidf.transform(val_texts)
X_train_tensor = torch.FloatTensor(X_train_tfidf.toarray()).to(device)
y_train_tensor = torch.LongTensor(train_labels_enc).to(device)
X_val_tensor = torch.FloatTensor(X_val_tfidf.toarray()).to(device)

class MCQNet(nn.Module):
    def __init__(self, input_dim):
        super(MCQNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 128), nn.ReLU(), nn.Linear(128, 5)
        )
    def forward(self, x): return self.net(x)
        
model_scratch = MCQNet(X_train_tensor.shape[1]).to(device)
optimizer = torch.optim.AdamW(model_scratch.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=32, shuffle=True)
model_scratch.train()

for epoch in range(15):
    for bx, by in train_loader:
        optimizer.zero_grad()
        loss = criterion(model_scratch(bx), by)
        loss.backward()
        optimizer.step()
model_scratch.eval()

with torch.no_grad():
    scratch_val_preds = torch.argmax(model_scratch(X_val_tensor), dim=1).cpu().numpy()
acc_scratch = accuracy_score(val_labels_enc, scratch_val_preds)
f1_scratch = f1_score(val_labels_enc, scratch_val_preds, average='weighted')

wandb.log({"model": "MLP_Scratch", "accuracy": acc_scratch, "f1": f1_scratch})
wandb.finish()



# Model 2

In [ ]:
# Model 2
wandb.init(project="23f3003059-t22026", name="DistilBERT_Pretrained", config={"metric": "MAP@3"})

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
def tokenize_fn(ex):
    return tokenizer(ex["text"], padding="max_length", truncation=True, max_length=512)
    
train_ds = Dataset.from_dict({"text": train_texts, "labels": train_labels_enc}).map(tokenize_fn, batched=True)
val_ds = Dataset.from_dict({"text": val_texts, "labels": val_labels_enc}).map(tokenize_fn, batched=True)
train_ds.set_format("torch"); val_ds.set_format("torch")

model_pretrained = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=5)

args = TrainingArguments(
    output_dir="./res", 
    num_train_epochs=3, 
    per_device_train_batch_size=16, 
    eval_strategy="epoch", 
    save_strategy="epoch", 
    load_best_model_at_end=True, 
    metric_for_best_model="accuracy", 
    report_to="wandb", 
    fp16=True
)


def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    return {
        "accuracy": accuracy_score(p.label_ids, preds),
        "f1": f1_score(p.label_ids, preds, average='weighted')
    }

trainer = Trainer(
    model=model_pretrained,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)
trainer.train()

eval_results = trainer.evaluate()
wandb.log({"model": "DistilBERT_Pretrained", "accuracy": eval_results["eval_accuracy"], "f1": eval_results["eval_f1"]})
wandb.finish()

# Model 3

# Model 3 Additional model
wandb.init(project="23f3003059-t22026", name="XGBoost_Additional", config={"metric": "MAP@3"})

xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42)
xgb_model.fit(X_train_tfidf, train_labels_enc)
xgb_val_preds = xgb_model.predict(X_val_tfidf)
acc_xgb = accuracy_score(val_labels_enc, xgb_val_preds)
f1_xgb = f1_score(val_labels_enc, xgb_val_preds, average='weighted')

wandb.log({"model": "XGBoost_Additional", "accuracy": acc_xgb, "f1": f1_xgb})
wandb.finish()